# 🚀 CryptoFL — Scaling Experiment na GPU (Google Colab)

Roda o `scaling_experiment.py` (FedAvg vs FedProx × nº de clientes) usando **GPU**.
O código detecta CUDA sozinho (`flower_fl/client.py`), então em GPU o CIFAR/ResNet-18
cai de **~30 min/round para segundos** — e somem os OOM / *ping timeout* que travavam em CPU.

**Antes de tudo:** menu **Runtime → Change runtime type → Hardware accelerator → GPU (T4)** → *Save*.

Depois rode as células em ordem (Runtime → Run all).


## 0) Confirmar que a GPU está ativa


In [ ]:
import torch, subprocess
print('torch:', torch.__version__, '| CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\u26a0\ufe0f  SEM GPU! Runtime > Change runtime type > GPU e rode de novo.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


## 1) Clonar o repositório + instalar dependências
Repo **público** → clone direto. O torch/CUDA já vêm no Colab; só falta o Flower.

> \u26a0\ufe0f NÃO troque o branch: as correções estão em **`claude/loving-hypatia-f54mj0`**, não no `main`.
> Sobre o `protobuf`: o flwr 1.7.0 fixa `protobuf<5`, mas o **grpcio do Colab (>=1.71) exige
> `protobuf>=5.26`** — com a versão 4.x os clientes morrem no startup. Por isso forçamos 5.x
> abaixo. O aviso `flwr requires protobuf<5` é **inofensivo** (o pipeline roda com 5.x).


In [ ]:
import os, shutil
os.chdir('/content')
shutil.rmtree('cryptoFL', ignore_errors=True)   # garante o branch certo mesmo se a pasta já existir
!git clone -b claude/loving-hypatia-f54mj0 https://github.com/Luanmantegazine/cryptoFL.git
%cd /content/cryptoFL
print('commit atual (deve ser das correcoes/notebook):')
!git log --oneline -1
!pip install -q 'flwr==1.7.0' 'numpy<2.0'
# grpcio do Colab (>=1.71) exige protobuf>=5.26; o pip rebaixa pra 4.x ao instalar o flwr,
# e aí os CLIENTES quebram no import dos _pb2 do grpc. Forçamos 5.x (aviso do flwr é inofensivo).
!pip install -q -U 'protobuf>=5.26,<6'
import google.protobuf as _pb; print('\u2705 ambiente pronto | protobuf', _pb.__version__)


## 2) Teste rápido (~1 min) — confirma que conecta e usa a GPU
2 clientes, 1 round, MNIST. Se terminar com acurácia, o pipeline está OK na GPU.
Se aparecer `[WARN] clients died right after spawn`, quase sempre é protobuf (rode a célula 1 de novo).


In [ ]:
!KMP_DUPLICATE_LIB_OK=TRUE python scaling_experiment.py --clients-list 2 --rounds 1 --aggregators fedavg --repetitions 1 --dataset mnist --model mnistnet --output-dir results/_smoke


## 3) ⭐ Sweep de escalabilidade — MNIST (figura principal, ~15\u201325 min)
N = 2,4,6,8,10 × FedAvg/FedProx × 3 reps. É a figura central (escala + variância + agregadores).


In [ ]:
!mkdir -p logs && KMP_DUPLICATE_LIB_OK=TRUE python scaling_experiment.py --clients-list 2,4,6,8,10 --rounds 3 --aggregators fedavg,fedprox --repetitions 3 --dataset mnist --model mnistnet --output-dir results/scaling_mnist 2>&1 | tee logs/scaling_mnist.log


## 4) Ponto único de generalização — CIFAR-10 / ResNet-18 (~5\u201310 min)
**Não é sweep.** Só N=2 (1 server + 2 clientes) para mostrar que treina CIFAR não-IID com
ResNet-18. Cada processo come ~1,5\u20132 GB de RAM; em N alto o Colab free (~13 GB) estoura —
por isso o CIFAR fica em N=2. `--min-fraction 0.8` evita travar se um cliente cair.

> Quer um pouco mais de acurácia? `--rounds 2`. NÃO aumente N aqui (é um ponto, não uma curva).


In [ ]:
# pré-baixa o CIFAR-10 (uma vez) para os clientes não correrem pelo download
import torchvision
torchvision.datasets.CIFAR10('./data', train=True, download=True)
torchvision.datasets.CIFAR10('./data', train=False, download=True)
print('\u2705 CIFAR-10 em ./data')


In [ ]:
!mkdir -p logs && KMP_DUPLICATE_LIB_OK=TRUE python scaling_experiment.py --clients-list 2 --rounds 1 --aggregators fedavg,fedprox --repetitions 1 --alpha 0.1 --dataset cifar10 --model resnet18 --min-fraction 0.8 --output-dir results/cifar_point 2>&1 | tee logs/cifar_point.log


## 5) Ver resultados (tabela SUMMARY + figuras)


In [ ]:
import json
from IPython.display import Image, display

def summary(path):
    s = json.load(open(path)); cfg = s['config']; r = s['results']
    print(f"== {cfg['dataset'].upper()} | {cfg['aggregators']} | N={cfg['clients_list']} | reps={cfg['repetitions']} ==")
    h = f"{'agg':<9}{'N':>4}{'mean_acc':>11}{'std_acc':>10}{'mean_time_s':>13}{'n_runs':>8}"
    print(h); print('-'*len(h))
    for n in cfg['clients_list']:
        for a in cfg['aggregators']:
            e = r.get(str(n), {}).get(a)
            if e:
                print(f"{a:<9}{n:>4}{e['mean_accuracy']:>11.4f}{e['std_accuracy']:>10.4f}{e['mean_time_s']:>13.2f}{e['n_runs']:>8}")

for tag in ['results/scaling_mnist', 'results/cifar_point']:
    p = f'{tag}/scaling_summary.json'
    try:
        summary(p)
        for fig in ['scaling_accuracy.png', 'scaling_time.png']:
            try: display(Image(f'{tag}/{fig}'))
            except Exception: pass
    except FileNotFoundError:
        print(f'(sem {p} ainda)')
    print()


## 6) Salvar os resultados (a sessão do Colab é efêmera!)
**Opção A** — baixar um zip. **Opção B** (célula seguinte) — salvar no Google Drive (persiste).


In [ ]:
!zip -qr resultados.zip results logs && echo 'zip pronto'
from google.colab import files
files.download('resultados.zip')


In [ ]:
# Opção B \u2014 Google Drive (descomente para usar):
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results logs '/content/drive/MyDrive/cryptoFL_results' && echo 'salvo no Drive'
